<a href="https://colab.research.google.com/github/YuXuan20040221/GDpj/blob/fish/RoadDamage_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 道路安全

注意事項:

* 需要用到GPU到 執行階段 > 變更執行階段類型去設定
* 打 "*" 號的地方是剛開始(打開colab)時需要執行的
* 要換資料集到"資料集設定" (旁邊的目錄有寫)
* 設定修改之後要先跑一次讓他寫進檔案才有效
* 調整參數到"訓練"


# *掛載 Google Drive
讓 Colab 讀取放在雲端硬碟的資料集
(這邊不用動)
/content/drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# *安裝相關套件

* YOLO套件

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.4 MB/s eta 0:00:00


## YOLO套件路徑
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules

/usr/local/lib/python3.11/dist-packages/ultralytics/models

# 設定(yaml)

## 資料集設定:

自行修改資料集目錄路徑

In [ ]:
# 定義train.yaml ->
yaml_content = """
path: /content/drive/MyDrive/datasets  # 整體資料集根目錄
train: Choose/train_all/images # 訓練用資料夾位置
val: Choose/valid/images # 驗證用資料夾位置

nc: 1 # 類別個數
names: ["pothole"] # 類別名稱
"""

# 寫入雲端 ->
with open("/content/drive/MyDrive/datasets/trainYOLO.yaml", "w") as f:
    f.write(yaml_content)

# 訓練
參數調整參考:
[YOLO調整指南](https://docs.ultralytics.com/zh/guides/hyperparameter-tuning/)

In [ ]:
import os
import torch
import sys
import torch.nn as nn
from ultralytics import YOLO

# 建立模型

# 模型名稱: 可用YOLO原始模型、自定義模型、之前練過的
# model = YOLO("yolov8n.pt")



# 我自己要玩的，不想用就幫我註解掉就好(刪了我會哭嗚嗚)

# 1. 自訂模型架構
# model = YOLO("/content/drive/MyDrive/model/Preprocessing.yaml")
model = YOLO("/content/drive/MyDrive/model/model_Preprocessing/yolov8n_Preprocessing_12/weights/last.pt")

# # 2. 官方模型(抓預訓練權重用)
# # pretrained = YOLO("yolov8n.pt")
# pretrained = YOLO("/content/drive/MyDrive/model/model_Preprocessing/yolov8n_Preprocessing_1/weights/best.pt")

# # 3. 取得預訓練模型的state_dict
# pretrained_dict = pretrained.model.state_dict()
# new_model_dict = model.model.state_dict()

# # 4. 把對得上的 key 丟進去（只留 backbone 的部分）
# filtered_dict = {k: v for k, v in pretrained_dict.items() if k in new_model_dict and v.shape == new_model_dict[k].shape}

# # 5. 套用進去
# model.model.load_state_dict(filtered_dict, strict=False)

# 訓練模型
model.train(
    # 訓練用參數
    data="/content/drive/MyDrive/datasets/trainYOLO.yaml",
    epochs=800, # 圈數
    batch=32, # 單次處理影像數量
    imgsz=640, # 圖片大小
    project="/content/drive/MyDrive/model/model_Preprocessing", # 存在哪資料夾
    name="yolov8n_Preprocessing_1", # 叫啥名
    lr0=0.0005, # 初始學習率(越小收斂速度越慢但越穩)
    lrf=0.002, # 最終學習率(是lr0的幾%，控制下降幅度)
    weight_decay=0.0005, # L2，防overfitting(但調太大模型難學)
    optimizer="AdamW", # 優化器(AdamW好像大家都會用)
    patience=100,  # earlystopping
    seed=42,
    warmup_epochs=50, # 穩定前幾epoch
    # dropout=0.2, # 關掉多少神經元
    device=0,
    # freeze=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22],
    resume=True, # 繼續訓練
    # ema=True,
    mosaic=0.2,

    # 圖片處理
    hsv_h=0.015,     # 色相擾動
    hsv_s=0.7,       # 飽和度擾動
    hsv_v=0.4,       # 亮度擾動
    degrees=10.0,    # 旋轉
    translate=0.1,   # 平移
    scale=0.3,       # 縮放
    shear=5.0,       # 剪切
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.196 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/datasets/trainYOLO.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=800, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False,

/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:2203: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  A = torch.tensor(A, dtype=src.dtype, device=src.device) # 使用 src 的 dtype


src:
 tensor([[nan, nan],
        [nan, nan],
        [nan, nan],
        [nan, nan]])
Preprocessing summary: 150 layers, 4,096,529 parameters, 4,096,513 gradients, 10.8 GFLOPs

Transferred 379/379 items from pretrained weights
Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.5±0.1 ms, read: 0.6±0.5 MB/s, size: 319.9 KB)
train: Scanning /content/drive/MyDrive/datasets/Choose/train_all/labels... 1400 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1400/1400 1.9it/s 12:16
train: /content/drive/MyDrive/datasets/Choose/train_all/images/G0064103.jpg: 2 duplicate labels removed
train: /content/drive/MyDrive/datasets/Choose/train_all/images/G0064409.jpg: 4 duplicate labels removed
train: /content/drive/MyDrive/datasets/Choose/train_all/images/G0064417.jpg: 6 duplicate labels removed
train: /content/drive/MyDrive/datasets/Choose/train_all/images/G0064419.jpg: 4 duplicate labels removed


# 實驗

## SpatialTransformer

In [ ]:
# from ultralytics.nn.modules.block import Preprocessing
import torchvision.utils as vutils
import os

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super(h_sigmoid, self).__init__()
        self.relu = nn.ReLU6(inplace=inplace)

    def forward(self, x):
        return self.relu(x + 3) / 6

class SELayer(nn.Module):
    def __init__(self, channel, reduction=4):
        super(SELayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel),
            h_sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x)
        y = y.view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class LightCorrection(nn.Module):
    def __init__(self, num_channels=3):
        super().__init__()
        self.a = nn.Parameter(torch.ones(1, num_channels, 1, 1, dtype=torch.float))
        self.b = nn.Parameter(torch.zeros(1, num_channels, 1, 1, dtype=torch.float))

    def forward(self, x):
        return self.a * x + self.b

class SpatialTransformer(nn.Module):
    def __init__(self, c1, reg_weight=0.01):
        super(SpatialTransformer, self).__init__()

        # localization network（提取特徵來預測變換矩陣）
        self.localization = nn.Sequential(
            nn.Conv2d(c1, 16, kernel_size=7, stride=2, padding=3),
            nn.ReLU(True),
            nn.Conv2d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 多加一層
            nn.ReLU(True),
            SELayer(channel=64),  # attention 強化重要特徵
            nn.AdaptiveAvgPool2d((16, 16))  # 固定輸出大小
        )

        self._fixed_flattened_size = 64 * 16 * 16  # C * H * W

        # FC layers -> output affine transform (6 params)
        self.fc_loc = nn.Sequential(
            nn.Linear(self._fixed_flattened_size, 64),
            nn.ReLU(True),
            nn.Linear(64, 6)
        )

        # 初始化為 identity transform
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(
            torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float)
        )

        # regularization 權重（懲罰離 identity 太遠）
        self.reg_weight = reg_weight

    def stn(self, x):
        xs = self.localization(x)
        xs = xs.view(-1, self._fixed_flattened_size)

        theta = self.fc_loc(xs)
        theta = theta.view(-1, 2, 3)

        # 生成 grid 並做仿射變換
        grid = F.affine_grid(theta, x.size(), align_corners=True)
        x_transformed = F.grid_sample(x, grid, align_corners=True)

        return x_transformed, theta

    def forward(self, x):
        x_transformed, theta = self.stn(x)
        return x_transformed, theta

    def regularization_loss(self, theta):
        """
        θ 正則化: 讓 affine matrix 不要離 identity 太遠
        """
        identity = torch.tensor([[1, 0, 0],
                                 [0, 1, 0]], dtype=torch.float,
                                device=theta.device)
        identity = identity.unsqueeze(0).expand_as(theta)  # [B, 2, 3]
        return F.mse_loss(theta, identity) * self.reg_weight


class Preprocessing(nn.Module):
    def __init__(self, c1, c2):
        super(Preprocessing, self).__init__()
        self.save_dir = "/content/drive/MyDrive/model/model_Preprocessing/preprocessed"
        self.STN = PerspectiveSTN(c1)
        self.LC = LightCorrection(num_channels=c1)
        self.filter = nn.Conv2d(c1, c2, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        y_stn = self.STN(x)
        # y_lc = self.LC(y_stn)
        output = self.filter(y_stn)

        if self.save_dir is not None and not self.training and y_stn.shape[1] == 3:
            try:
                vutils.save_image(
                    (y_stn - y_stn.min()) / (y_stn.max() - y_stn.min() + 1e-6),  # normalize to 0~1
                    f"{self.save_dir}/preprocessed_batch.png"
                )
            except TypeError as e:
                print(f"Warning: Could not save image due to TypeError: {e}. Data shape: {output.shape}, dtype: {output.dtype}")

        return output

# # 假設輸入圖片是 batch=2, channel=3, size=128x128
# x = torch.randn(1, 3, 128, 128)

# # 測試 Preprocessing
# model = Preprocessing(c1=3, c2=3)
# y = model(x)

# print("input shape:", x.shape)
# print("output shape:", y.shape)

### PerspectiveSTN-1

In [3]:
class PerspectiveSTN(nn.Module):
    def __init__(self, c1):
        super(PerspectiveSTN, self).__init__()

        self.localization = nn.Sequential(
            nn.Conv2d(c1, 16, kernel_size=7, stride=2, padding=3),
            nn.ReLU(True),
            nn.Conv2d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            CoordAtt(64, 64, 8),
            nn.AdaptiveAvgPool2d((16, 16))
        )

        self._fixed_flattened_size = 64 * 16 * 16  # C * H * W

        self.fc_loc = nn.Sequential(
            nn.Linear(self._fixed_flattened_size, 64),
            nn.ReLU(True),
            nn.Linear(64, 8),
            nn.Tanh(),
            nn.ReLU(True),
        )

        # (Δx, Δy)
        # self.fc_loc[2].weight.data.zero_()
        nn.init.normal_(self.fc_loc[2].weight, mean=0., std=.01) # 例如，使用均值 0，標準差 0.01 的正態分佈
        self.fc_loc[2].bias.data.copy_(torch.zeros(8))

    def get_perspective_transform(self, src, dst):
        """計算 homography matrix H 並求 H 的逆矩陣 H_inv"""
        B = src.size(0)
        H_inv_list = [] # 這裡我們要收集 H_inv

        for b in range(B):
            A = []
            if torch.isnan(src[b]).any() or torch.isinf(src[b]).any() or torch.isnan(dst[b]).any() or torch.isinf(dst[b]).any():
                 print(f"Warning: NaN or Inf detected in src or dst points for batch {b} before forming matrix A! Returning Identity H_inv.")
                 identity_inv = torch.eye(3, device=src.device, dtype=src.dtype) # 使用 src 的 dtype
                 H_inv_list.append(identity_inv)
                 continue # 跳過後面的計算

            for (x, y), (xp, yp) in zip(src[b], dst[b]):
                A.append([-x, -y, -1, 0, 0, 0, x * xp, y * xp, xp])
                A.append([0, 0, 0, -x, -y, -1, x * yp, y * yp, yp])
            A = torch.tensor(A, dtype=src.dtype, device=src.device) # 使用 src 的 dtype

            # 檢查 A 是否包含 NaN 或 Inf
            if torch.isnan(A).any() or torch.isinf(A).any():
                 print(f"Warning: NaN or Inf detected in matrix A for batch {b}! Returning Identity H_inv.")
                 identity_inv = torch.eye(3, device=src.device, dtype=src.dtype) # 使用 src 的 dtype
                 H_inv_list.append(identity_inv)
                 continue # 跳過後面的計算

            try:
                # 計算 H
                _, _, V = torch.linalg.svd(A)
                h = V[-1, :]
                H = h.view(3, 3)

                # 檢查 H 是否有問題
                if torch.isnan(H).any() or torch.isinf(H).any() or H[2, 2] == 0:
                     print(f"Warning: NaN, Inf or zero in H[2,2] detected in Homography matrix H for batch {b}! Returning Identity H_inv.")
                     identity_inv = torch.eye(3, device=src.device, dtype=src.dtype) # 使用 src 的 dtype
                     H_inv_list.append(identity_inv)
                     continue # 跳過後面的計算

                # 確保 H[2, 2] 不是太小
                if torch.abs(H[2, 2]) < 1e-8:
                          print(f"Warning: H[2,2] is too small ({H[2, 2].item()}) for batch {b}! Returning Identity H_inv.")
                          identity_inv = torch.eye(3, device=src.device, dtype=src.dtype) # 使用 src 的 dtype
                          H_inv_list.append(identity_inv)
                          continue # 跳過後面的計算

                # 標準化 H 並計算 H_inv
                H = H / H[2, 2]
                H_inv = torch.inverse(H)

                # 檢查 H_inv 是否為 NaN or Inf
                if torch.isnan(H_inv).any() or torch.isinf(H_inv).any():
                     print(f"Warning: NaN or Inf detected in H_inv for batch {b}! H: {H}. Returning Identity H_inv.")
                     identity_inv = torch.eye(3, device=src.device, dtype=src.dtype) # 使用 src 的 dtype
                     H_inv_list.append(identity_inv)
                else:
                    H_inv_list.append(H_inv)


            except Exception as e:
                 print(f"Warning: Error during SVD, Homography calculation or Inverse for batch {b}: {e}! Returning Identity H_inv.")
                 identity_inv = torch.eye(3, device=src.device, dtype=src.dtype) # 使用 src 的 dtype
                 H_inv_list.append(identity_inv)


        return torch.stack(H_inv_list, dim=0)

    def forward(self, x):
        B, C, H_in, W_in = x.size()
        dtype = x.dtype

        # 初始化 (左上, 右上, 右下, 左下)
        base_corners = torch.tensor([
            [0, 0],
            [W_in - 1, 0],
            [W_in - 1, H_in - 1],
            [0, H_in - 1]
        ], dtype=torch.float32, device=x.device).unsqueeze(0).repeat(B, 1, 1)  # [B, 4, 2]

        xs = self.localization(x)
        xs = xs.view(-1, self._fixed_flattened_size)

        offsets = self.fc_loc(xs)
        offsets = offsets.view(-1, 4, 2)

        offset_scale = [W_in / 2.0, H_in / 2.0]

        src = base_corners.clone()
        # 左上
        src[:, 0, 0] += offsets[:, 0, 0] * offset_scale[0]
        src[:, 0, 1] += offsets[:, 0, 1] * offset_scale[1]
        # 右上
        src[:, 1, 0] -= offsets[:, 1, 0] * offset_scale[0]
        src[:, 1, 1] += offsets[:, 1, 1] * offset_scale[1]
        # 右下
        src[:, 2, 0] -= offsets[:, 2, 0] * offset_scale[0]
        src[:, 2, 1] -= offsets[:, 2, 1] * offset_scale[1]
        # 左下
        src[:, 3, 0] += offsets[:, 3, 0] * offset_scale[0]
        src[:, 3, 1] -= offsets[:, 3, 1] * offset_scale[1]

        dst = base_corners.clone()

        print("src:\n", src[0,:,:])

        H_inv = self.get_perspective_transform(src, dst)  # [B, 3, 3]

        yy, xx = torch.meshgrid(
            torch.arange(H_in, device=x.device, dtype=dtype),
            torch.arange(W_in, device=x.device, dtype=dtype),
            indexing="ij"
        )
        ones = torch.ones_like(xx, dtype=dtype)
        grid = torch.stack([xx, yy, ones], dim=-1).view(-1, 3)
        grid = grid.T.unsqueeze(0).repeat(B, 1, 1)

        mapped = torch.matmul(H_inv.to(dtype), grid)
        mapped = mapped / (mapped[:, 2:3, :] + 1e-8)

        x_norm = 2.0 * mapped[:, 0, :] / (W_in - 1) - 1
        y_norm = 2.0 * mapped[:, 1, :] / (H_in - 1) - 1
        sampling_grid = torch.stack([x_norm, y_norm], dim=-1)
        sampling_grid = sampling_grid.view(B, H_in, W_in, 2)

        out = F.grid_sample(x, sampling_grid, align_corners=True)

        return out

NameError: name 'nn' is not defined

## CA

In [ ]:
# from ultralytics.nn.modules.block import CoordAtt
class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super(h_sigmoid, self).__init__()
        self.relu = nn.ReLU6(inplace=inplace)

    def forward(self, x):
        return self.relu(x + 3) / 6

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super(h_swish, self).__init__()
        self.sigmoid = h_sigmoid(inplace=inplace)

    def forward(self, x):
        return x * self.sigmoid(x)

class CoordAtt(nn.Module):
    def __init__(self, inp, oup, reduction=32):
        super(CoordAtt, self).__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mip = max(8, inp // reduction)

        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = h_swish()

        self.conv_h = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, oup, kernel_size=1, stride=1, padding=0)


    def forward(self, x):
        identity = x

        n,c,h,w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()

        out = identity * a_w * a_h

        return out

# 模型評估

In [ ]:
from ultralytics import YOLO
import os
import torch
import torchvision.utils as vutils

def save_preprocessed_images(images, save_dir="runs/preprocessed", step=0):
    # images: Tensor [B, C, H, W]
    vutils.save_image(images, f"{save_dir}/batch_{step}.png", nrow=4, normalize=True)


# 載入訓練好的模型 (請確認路徑正確)
# 你可以使用 best.pt 或 last.pt
model_path = "/content/drive/MyDrive/model/model_Preprocessing/yolov8n_Preprocessing_14/weights/best.pt"
model = YOLO(model_path)

# 定義測試資料集的路徑 (請確認路徑和 trainYOLO.yaml 中的 val 路徑一致)
# 如果你的測試資料集是獨立的，需要另外建立一個 yaml 檔案來指定測試資料夾
# 這裡先使用 trainYOLO.yaml 中的 val 路徑作為示範
data_yaml_path = "/content/drive/MyDrive/datasets/trainYOLO.yaml"


# 在測試資料集上進行驗證
# 你可以使用 split='test' 如果你的 yaml 檔案中有定義測試集
# 這裡使用 split='val' 因為 trainYOLO.yaml 中只有 train 和 val
results = model.val(data=data_yaml_path, split='val')

# 打印評估結果
print("\n--- 模型評估結果 ---")
print(f"mAP50-95: {results.box.map}")
print(f"mAP50: {results.box.map50}")
print(f"Precision: {results.box.mp}")
print(f"Recall: {results.box.mr}")
print(f"F1 Score: {(2 * results.box.mp * results.box.mr) / (results.box.mp + results.box.mr) if (results.box.mp + results.box.mr) > 0 else 0}") # 手動計算 F1 Score

# 打印模型性能資訊
print("\n--- 模型性能資訊 ---")
# 模型參數數量
# Directly calculate trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型可訓練參數數量 (trainable parameters): {trainable_params}")

# Total model parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"模型總參數數量 (total parameters): {total_params}")


# GFLOPs (在 val 時會計算並顯示在上面的輸出中)
print("GFLOPs (請參考上方 val 執行輸出的 summary 部分)")

# 模型大小 (通過文件大小估計)
model_file_size = os.path.getsize(model_path) / (1024 * 1024) # 轉為 MB
print(f"模型檔案大小: {model_file_size:.2f} MB")

print("\n--- 評估結束 ---")
